# 13 · Final Results

**⚠️  This is the only notebook that uses the held-out test set.**
Do not load `get_test_loader()` anywhere else.

Aggregates all experiment results into unified comparison tables and plots.
Fill in checkpoint paths after completing notebooks 01–12.


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Scratch, VGG_Pretrained,
    ResNet_Scratch, ResNet18_Pretrained,
    MobileNetV2_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, MODEL_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_model, train_model_three_phase,
    train_multi_seed, train_kd, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
# ── Test loader — used ONLY here ─────────────────────────────────────
test_loader = get_test_loader(batch_size=64)
_, val_loader = get_loaders(batch_size=64)


In [ ]:
# ── All checkpoints — fill in after running 01–12 ────────────────────
ALL_MODELS = [
    # (label, model_fn, checkpoint_path, phase)
    # Teachers
    ("VGG (scratch)",           VGG_Scratch,          f"{SAVE_DIR}/vgg_scratch_seed_XX.pth",          "Teacher"),
    ("VGG16-BN (pretrained)",   VGG_Pretrained,        f"{SAVE_DIR}/vgg_pretrained_seed_XX.pth",       "Teacher"),
    ("ResNet (scratch)",        ResNet_Scratch,         f"{SAVE_DIR}/resnet_scratch_seed_XX.pth",       "Teacher"),
    ("ResNet18 (pretrained)",   ResNet18_Pretrained,    f"{SAVE_DIR}/resnet18_pretrained_seed_XX.pth",  "Teacher"),
    # Students baseline
    ("MobileNetV2 (scratch)",          MobileNetV2_Scratch,    f"{SAVE_DIR}/mobilenetv2_baseline_seed_XX.pth", "Student"),
    ("MobileNetV3-Small (pretrained)", MobileNetV3_Pretrained, f"{SAVE_DIR}/mobilenetv3_baseline_seed_XX.pth", "Student"),
    # KD
    ("MobileNetV3 + KD-FT",     MobileNetV3_Pretrained, f"{SAVE_DIR}/kd_ft_seed_XX.pth",     "KD"),
    ("MobileNetV3 + KD-Scratch",MobileNetV3_Pretrained, f"{SAVE_DIR}/kd_scratch_seed_XX.pth","KD"),
    # Pruned
    ("MobileNetV3 + KD + Pruned 30%", MobileNetV3_Pretrained, f"{SAVE_DIR}/pruned_30pct.pth", "Pruned"),
]


In [ ]:
def measure_latency(model, device, runs=200, warmup=20):
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    with torch.no_grad():
        for _ in range(warmup): model(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(runs): model(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / runs * 1000

rows = []
for label, fn, ckpt, phase in ALL_MODELS:
    if not os.path.exists(ckpt):
        print(f"⚠️  Skipping: {label}"); continue
    m = fn().to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    val_acc  = evaluate(m, val_loader,  device)
    test_acc = evaluate(m, test_loader, device)
    tp, _  = count_params(m)
    size   = model_size_mb(m)
    lat    = measure_latency(m, device)
    rows.append({
        "Model": label, "Phase": phase,
        "Val Acc (%)":  round(val_acc*100,  2),
        "Test Acc (%)": round(test_acc*100, 2),
        "Params (M)":   round(tp/1e6,       2),
        "Size (MB)":    round(size,          2),
        "Latency (ms)": round(lat,           3),
    })
    print(f"{label:42s}  val={val_acc*100:.2f}%  test={test_acc*100:.2f}%")


In [ ]:
import pandas as pd
df = pd.DataFrame(rows).set_index("Model")
print("\n" + "="*80)
print("FINAL RESULTS — TEST SET")
print("="*80)
print(df.drop(columns=["Phase"]).to_string())


In [ ]:
# ── Accuracy comparison chart ─────────────────────────────────────────
phase_colors = {"Teacher": "#4878D0", "Student": "#55A868", "KD": "#E8735A", "Pruned": "#C44E52"}
colors = [phase_colors[r["Phase"]] for r in rows]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(df.index, df["Test Acc (%)"], color=colors, edgecolor="white")
ax.set_ylabel("Test Accuracy (%)"); ax.set_title("Final Results — Test Set Accuracy")
ax.set_ylim(df["Test Acc (%)"].min() - 3, 100)
for p in bars:
    ax.annotate(f"{p.get_height():.2f}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=7)

from matplotlib.patches import Patch
legend = [Patch(color=c, label=l) for l, c in phase_colors.items()]
ax.legend(handles=legend, loc="lower right")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()


In [ ]:
# ── Accuracy vs Model Size scatter ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for _, row in df.iterrows():
    ax.scatter(row["Size (MB)"], row["Test Acc (%)"],
               color=phase_colors.get(row["Phase"], "gray"), s=80, zorder=3)
    ax.annotate(row.name, (row["Size (MB)"], row["Test Acc (%)"]),
                textcoords="offset points", xytext=(5, 3), fontsize=7)
ax.set_xlabel("Model Size (MB)"); ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Accuracy vs Model Size")
ax.legend(handles=legend); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── KD improvement summary ────────────────────────────────────────────
print("\nKnowledge Distillation Impact")
print("─"*50)
if "MobileNetV3-Small (pretrained)" in df.index and "MobileNetV3 + KD-FT" in df.index:
    baseline = df.loc["MobileNetV3-Small (pretrained)", "Test Acc (%)"]
    kd_ft    = df.loc["MobileNetV3 + KD-FT",           "Test Acc (%)"]
    print(f"  Student baseline (no KD):  {baseline:.2f}%")
    print(f"  Student + KD-FT:           {kd_ft:.2f}%")
    print(f"  Improvement:               +{kd_ft - baseline:.2f}%")

print("\nQuantization Impact (see notebook 12 for ONNX accuracy)")
print("─"*50)
print("  FP32 ONNX:  see notebook 12")
print("  INT8 ONNX:  see notebook 12")
print("  STM32 FP32: fill in after hardware validation")
print("  STM32 INT8: fill in after hardware validation")
